In [41]:
import matplotlib.pyplot as plt

In [ ]:
import torch
from einops import rearrange

def stratified_window(
    logits, 
    num_samples: int, 
    left: float = 0.0, 
    right: float = 1.0, 
    is_probs: bool = False
):
    """
    Args:
        logits: (Batch, Seq, Vocab)
        num_samples: How many samples to draw from the specified window.
        left: The start of the probability range (0.0 = most likely).
        right: The end of the probability range (1.0 = least likely).
    """
    left = min(max(left, 0.0), 1.0)
    right = min(max(right, 0.0), 1.0)
    assert left <= right, "Left boundary must be less than or equal to right boundary."
    
    if not is_probs:
        probs = torch.softmax(logits, dim=-1)
    else:
        probs = logits

    # 1. Sort probabilities descending so that 'left=0' is the most likely token
    probs_sorted, indices_sorted = torch.sort(probs, dim=-1, descending=True)
    
    # 2. Compute CDF on the sorted tokens
    cdf = probs_sorted.cumsum(dim=-1)

    # 3. Define the window width
    window_width = right - left
    
    # 4. Generate stratified noise within the [left, right] range
    # If num_samples=1, left=0.1, right=0.2:
    # u will be in [0, 1], then scaled to [0, 0.1], then shifted to [0.1, 0.2]
    *batch_dims, seq_len, vocab_size = cdf.shape
    u = torch.rand(*batch_dims, seq_len, num_samples, device=logits.device)
    
    # This formula splits the window [left, right] into 'num_samples' equal strata
    # and picks one random point from each.
    strata_steps = torch.arange(num_samples, device=logits.device)
    strata = left + (strata_steps + u) / num_samples * window_width

    # 5. Find the indices in the sorted distribution
    # searchsorted finds the first index where cdf >= strata
    sampled_sorted_indices = torch.searchsorted(cdf, strata)
    sampled_sorted_indices = sampled_sorted_indices.clamp(max=vocab_size - 1)

    # 6. Map back to original vocabulary indices
    # indices_sorted is (Batch, Seq, Vocab), we need to gather from it
    # We need to expand indices_sorted to match the num_samples dimension
    # or use gather carefully.
    
    # Flatten batch/seq for easier gathering
    flat_indices_sorted = rearrange(indices_sorted, "b s v -> (b s) v")
    flat_sampled_indices = rearrange(sampled_sorted_indices, "b s n -> (b s) n")
    
    # Gather the actual token IDs
    samples = torch.gather(flat_indices_sorted, dim=1, index=flat_sampled_indices)
    
    return rearrange(samples, "(b s) n -> (b n) s", b=batch_dims[0])

In [59]:
batch, seq, vocab = 2, 5, 10

In [60]:
batches = []
for b in range(batch):
    seqes = []
    for s in range(seq):
        #seqes.append(torch.arange(start=vocab-1,end=-1,step=-1))
        seqes.append(torch.zeros(vocab))
    batches.append(torch.stack(seqes))
batches = torch.stack(batches)  # (Batch, Seq, Vocab)

In [61]:
batches

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]])

In [62]:
torch.softmax(batches.float(), dim=-1)[0][0]

tensor([0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000])

In [72]:
selected = stratified_window(batches.float(), num_samples=1, left=0.1, right=0.3)

In [77]:
selected.shape

torch.Size([2, 5])

In [75]:
import random

In [76]:
random.random()

0.6291844845070445

In [78]:
from torch.nn import functional as F

In [80]:
F.one_hot(selected, 10).shape

torch.Size([2, 5, 10])